In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rajmehra03/movielens100k/movies.csv
/kaggle/input/datasets/rajmehra03/movielens100k/ratings.csv
/kaggle/input/datasets/rajmehra03/movielens100k/u.data
/kaggle/input/datasets/rajmehra03/movielens100k/tags.csv
/kaggle/input/datasets/rajmehra03/movielens100k/links.csv


In [4]:
import tensorflow as tf

# 列出所有物理 GPU 设备
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"找到 GPU: {gpus}")
    for gpu in gpus:
        print(f"GPU 名称: {tf.config.experimental.get_device_details(gpu).get('device_name', 'unknown')}")
else:
    print("没有检测到 GPU，将使用 CPU")

找到 GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU 名称: Tesla T4
GPU 名称: Tesla T4


In [5]:
import os
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [60]:
tf.__version__

'2.19.0'

In [7]:
#data = tfds.load("movielens/100k-ratings", split="train")
#data.take(1)

In [6]:
import pandas as pd
path = "/kaggle/input/datasets/rajmehra03/movielens100k/ratings.csv"
df=pd.read_csv(path)
df.head()


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [7]:
#用户行为序列字典 {userid:{items':list, 'actions':list, 'time':list}}
def build_user_seq(df):
    user_seq={}
    for userid, group in df.groupby('userId'):
        group  = group.sort_values('timestamp')
        items = group['movieId'].tolist()
        time = group['timestamp'].tolist()
        actions = [1 if r>=4.0 else 0 for r in group['rating']]
        user_seq[userid] = {
            'items':items,
            'actions':actions,
            'times':time
        }
    return user_seq

user_seq = build_user_seq(df)

In [25]:
len(user_seq)

671

In [8]:
all_items = set(df['movieId'].unique())
item2id = {item : i+1 for i,item in enumerate(sorted(all_items))}
num_items = len(item2id) + 1 # 0 left for mask
num_actions = 2 # 0 and 1
print("all items cnt:", num_items)

all items cnt: 9067


In [9]:
def train_test_leave_one_split(user_seq):
    train_data = {}
    test_data = {}
    for userid, seq in user_seq.items():
        items = seq['items']
        actions = seq['actions']
        times = seq['times']
        if len(items) < 2:
            continue
        train_data[userid] = {
            'items':items[:-1],
            'actions':actions[:-1],
            'times':times[:-1]
        }
        test_data[userid] = {
            'items':[items[-1]],
            'actions':[actions[-1]],
            'times':[times[-1]]
        }
    return train_data, test_data

train_data,test_data = train_test_leave_one_split(user_seq)
    

In [13]:
x=[1,2,3,4]
y=[0]*3+x
print(y)

[0, 0, 0, 1, 2, 3, 4]


In [10]:
# Stochastic Length 随机截断，reduce compute and user behaviour temporally repetitive, 
# provide more possible sub-seqs for generalization ability

import numpy as np
def stochastic_sample(items, actions, times, max_len = 50, sl_prob = 0.3):
    if len(items) <= max_len:
        return items, actions, times
    
    if np.random.random() < sl_prob:
        start = np.random.randint(0, len(items) - max_len + 1)
        return items[start:start + max_len],actions[start:start+max_len],times[start:start+max_len]
    else:
        return items[-max_len:],actions[-max_len:],times[-max_len:]

In [13]:
def create_train_samples(train_data, max_len=50, sl_prob=0.3):
    X_items, X_actions, X_times = [], [], []
    y_items, y_actions = [], []
    for userid, data in user_seq.items():
        items = [item2id[i] for i in data['items']]
        actions = data['actions']
        times = data['times']

        for i in range(1, len(items)):
            in_items = items[:i]
            in_actions = actions[:i]
            in_times = times[:i]
            in_items,in_actions,in_times = stochastic_sample(in_items, in_actions, in_times, max_len, sl_prob)

            target_item = items[i]
            target_action = actions[i]

            # padding to maxlen
            pad_len = max_len - len(in_items)
            pad_items = [0] * pad_len + in_items
            pad_actions = [0] * pad_len + in_actions
            pad_times = [0] * pad_len + in_times

            X_items.append(pad_items)
            X_actions.append(pad_actions)
            X_times.append(pad_times)

            y_items.append(target_item)
            y_actions.append(target_action)

    X_items = np.array(X_items)
    X_actions = np.array(X_actions)
    X_times = np.array(X_times)
    y_items = np.array(y_items)
    y_actions = np.array(y_actions)

    return (X_items, X_actions, X_times), (y_items, y_actions)          

In [14]:
(X_train_items, X_train_actions, X_train_times), (y_train_items, y_train_actions) = create_train_samples(train_data, max_len=50, sl_prob=0.3)

In [15]:
X_train_items

array([[   0,    0,    0, ...,    0,    0, 1816],
       [   0,    0,    0, ...,    0, 1816, 1963],
       [   0,    0,    0, ..., 1816, 1963, 2926],
       ...,
       [   1, 1046,  980, ..., 4418, 2628, 4598],
       [1046,  980,  233, ..., 2628, 4598, 4611],
       [ 980,  233,  882, ..., 4598, 4611, 4697]])

In [16]:
def create_test_samples(train_data, test_data, max_len = 50):
    X_i, X_a, X_t, y_i, y_a = [], [], [], [], []
    for userid in test_data:
        if userid not in train_data:
            continue
        his = train_data[userid]
        test = test_data[userid]

        his_items =  [item2id[i] for i in his['items']]
        his_actions = his['actions']
        his_times = his['times']
        if len(his_items) > max_len:
            his_items =  his_items[-max_len:]
            his_actions = his_items[-max_len:]
            his_times = his_items[-max_len:]
        
        pad_len = max_len - len(his_items)

        X_i.append([0]*pad_len + his_items)
        X_a.append([0]*pad_len + his_actions)
        X_t.append([0]*pad_len + his_times)
        y_i.append(item2id[test['items'][0]])
        y_a.append(test['actions'][0])
    return (np.array(X_i), np.array(X_a), np.array(X_t)),(np.array(y_i), np.array(y_a))

(X_test_items, X_test_actions, X_test_times), (y_test_items, y_test_actions) = create_test_samples(train_data, test_data, max_len=50)

In [19]:
len(X_test_items)

671

In [17]:
split = train_test_split(X_train_items, X_train_actions, X_train_times, y_train_items, y_train_actions, test_size = 0.1, random_state = 33)

In [19]:
X_tr_i, X_val_i,X_tr_a, X_val_a,X_tr_t, X_val_t,y_tr_i,y_val_i,y_tr_a,y_val_a = split

In [20]:
print(len(X_tr_i),len(X_tr_a), len(X_val_i), len(X_val_a))

89399 89399 9934 9934


In [23]:
# batch_size=2
# x = np.array([[1.0,2.0,3.0],[1.1, 2.1, 3.1], [4.1,5.1,6.1],[4.1,5.1,6.1]])
# y = np.array([[1.0],[1.0],[0.0],[0.0]])
# ds = tf.data.Dataset.from_tensor_slices(({'items':x}, {'target':y}))
# ds=ds.shuffle(4)
# ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
# print(ds.element_spec)
# for batch in ds.take(1):
#     f,l=batch
#     for k,v in f.items():
#         print(f"{k}:{v.numpy()}")

In [21]:
batch_size=256

def make_dataset(X_i, X_a, X_t, y_i, y_a, shuffle = True):
    ds = tf.data.Dataset.from_tensor_slices(({'items':X_i, 'actions':X_a, 'times':X_t}, {'target_items':y_i, 'target_actions':y_a}))
    if shuffle:
        ds=ds.shuffle(10000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [22]:
train_ds = make_dataset(X_tr_i,X_tr_a, X_tr_t, y_tr_i, y_tr_a)
val_ds = make_dataset(X_val_i,X_val_a, X_val_t, y_val_i, y_val_a, shuffle=False)
test_ds = make_dataset(X_test_items, X_test_actions, X_test_times, y_test_items, y_test_actions, shuffle=False)

I0000 00:00:1773210469.091031      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773210469.096967      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [26]:
x = np.array([[1.0, 2.0, 3.0],[2.1, 4.1, 5.1]])
y = x[:, :, tf.newaxis] 
z = x[:, tf.newaxis, :]
print(y)
print("***********")
print(z)
print("--------------")
print(y-z)

[[[1. ]
  [2. ]
  [3. ]]

 [[2.1]
  [4.1]
  [5.1]]]
***********
[[[1.  2.  3. ]]

 [[2.1 4.1 5.1]]]
--------------
[[[ 0. -1. -2.]
  [ 1.  0. -1.]
  [ 2.  1.  0.]]

 [[ 0. -2. -3.]
  [ 2.  0. -1.]
  [ 3.  1.  0.]]]


In [27]:
x = y-z
e = tf.clip_by_value(x, -2, 2)
#print(e)
e = e + 2
pos_emb = keras.layers.Embedding(2*2+1, output_dim = 12, embeddings_initializer='uniform', name='pos_embedding')

emb = pos_emb(e)
print(e.shape)
print("***********")
print(emb.shape)

proj = keras.layers.Dense(1, use_bias=False, name='proj')
bias = proj(emb)
print(bias.shape)

bias =tf.squeeze(bias, axis=-1)
print(bias.shape)

(2, 3, 3)
***********
(2, 3, 3, 12)
(2, 3, 3, 1)
(2, 3, 3)


In [23]:
from tensorflow.keras import layers
class RelativePositionBias(layers.Layer):
    #相对位置偏置层
    def __init__(self, max_position = 128, hidden_size = 64):
        super().__init__()
        self.max_position = max_position
        self.hidden_size = hidden_size
        self.pos_emb = layers.Embedding(2 * max_position, output_dim = hidden_size, embeddings_initializer='uniform', name='pos_embedding')
        self.proj = layers.Dense(1, use_bias=False, name='pos_proj')

    def call(self, query_pos, key_pos):
        # self-attention same query key
        re_pos = query_pos[:,:,tf.newaxis] - key_pos[:, tf.newaxis, :] #[bs, seq_len, seq_len] 每一个seq内部每个位置和其他位置的距离
        re_pos = tf.clip_by_value(re_pos, -self.max_position, self.max_position)
        re_pos_id = re_pos + self.max_position #非负索引

        emb = self.pos_emb(re_pos_id) #[bs, seq_len, seq_len, hidden]
        bias = self.proj(emb) #[bs, seq_len, seq_len, 1]
        bias = tf.squeeze(bias, axis = -1) #[bs, seq_len, seq_len]
        return bias

In [29]:
x  = 1.0
logx = tf.math.log(x) / tf.math.log(2.0)
print(logx)
b = tf.floor(logx)
print(b)
print(tf.math.log(2.0))

tf.Tensor(0.0, shape=(), dtype=float32)
tf.Tensor(0.0, shape=(), dtype=float32)
tf.Tensor(0.6931472, shape=(), dtype=float32)


In [94]:
class TimeBias(layers.Layer):
    #时间偏置层，将时间diff映射到对数尺度桶中
    def __init__(self, max_buckets = 64, hidden_size = 64):
        super().__init__()
        self.max_buckets = max_buckets
        self.hidden_size = hidden_size
        self.time_emb = layers.Embedding(max_buckets, hidden_size, embeddings_initializer='uniform', name='time_embedding')
        self.proj = layers.Dense(1, use_bias=False, name='time_proj')

    def call(self, time_diffs):
        time_diffs = tf.cast(time_diffs, tf.float32)
        abs_diff = tf.maximum(tf.abs(time_diffs), 1e-6)
        log_diffs = tf.math.log(abs_diff) / tf.math.log(2.0)
        bucket_ids = tf.cast(tf.floor(log_diffs), tf.int32)
        bucket_ids = tf.clip_by_value(bucket_ids, 0, self.max_buckets - 1)

        emb = self.time_emb(bucket_ids)
        bias = self.proj(emb)
        bias = tf.squeeze(bias, axis = -1)
        return bias    

In [95]:
class CombineBias(layers.Layer):
    def __init__(self, max_position = 128, hidden_size = 64, max_buckets = 64,**kwargs):
        super().__init__()
        self.pos_bias = RelativePositionBias(max_position, hidden_size)
        self.time_bias = TimeBias(max_buckets, hidden_size)

    def call(self, query_pos, key_pos, time_diffs):
        return self.pos_bias(query_pos, key_pos) + self.time_bias(time_diffs)
        
        

In [12]:
import numpy as np
from tensorflow.keras import layers
x = np.array([[[1.0, 2.0, 3.0],[2.1, 4.1, 5.1]],[[1.1, 4.1, 5.1],[0.5, 0.8, 0.9]]])
print(x)
x = x[:, -1,: ]
print(x)
# y = layers.Dense(16, use_bias=False)(x)
# print(y)


[[[1.  2.  3. ]
  [2.1 4.1 5.1]]

 [[1.1 4.1 5.1]
  [0.5 0.8 0.9]]]
[[2.1 4.1 5.1]
 [0.5 0.8 0.9]]


In [39]:
for batch in train_ds:
    f,l=batch
    for k,v in f.items():
        print(v.shape)
        z = keras.layers.Embedding(num_items + 1, 64, mask_zero=True)(v)
        print(z.shape)
    break

(256, 50)
(256, 50, 64)
(256, 50)
(256, 50, 64)
(256, 50)
(256, 50, 64)


In [45]:
x = tf.range(10)
print(x)
y = x[tf.newaxis, :]
print(y)
z = tf.tile(y, [3, 2])
print(z)

tf.Tensor([0 1 2 3 4 5 6 7 8 9], shape=(10,), dtype=int32)
tf.Tensor([[0 1 2 3 4 5 6 7 8 9]], shape=(1, 10), dtype=int32)
tf.Tensor(
[[0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9]
 [0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9]
 [0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9]], shape=(3, 20), dtype=int32)


In [106]:
class HSTULayer(layers.Layer):
    def __init__(self, d_model = 64, num_heads = 8, max_position = 128, max_time_buckets = 64, dropout_rate = 0.0, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        assert self.head_dim * num_heads == d_model

        self.query = layers.Dense(d_model, use_bias = False)
        self.key = layers.Dense(d_model, use_bias = False)
        self.value = layers.Dense(d_model, use_bias = False)
        self.gate = layers.Dense(d_model, use_bias = False, activation='sigmoid')

        self.combined_bias = CombineBias(max_position, self.head_dim, max_time_buckets)
        self.out_proj = layers.Dense(d_model)
        self.att_drop = layers.Dropout(dropout_rate)
        self.ff_drop = layers.Dropout(dropout_rate)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    def split_heads(self, x, batch_size):
        #[bs, seq_len, d_model(emb_dim)]
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.head_dim)) #[bs, seq, numheads, head_dim]
        return tf.transpose(x, perm=[0, 2, 1, 3]) #[bs, numheads, seq, head_dim]方便每个head计算matmul

    def call(self, inputs, pos_ids=None, time_matrix=None, training=False):
        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]
        #input norm, 残差
        residual = inputs
        x = self.norm1(inputs)

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        U = self.gate(x)

        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size) #[bs, numheads, seq, head_dim]

        atten_scores = tf.matmul(Q, K, transpose_b=True) #[bs, numheads, seq, seq]
        atten_scores = atten_scores / tf.math.sqrt(tf.cast(self.head_dim, tf.float32))

        if pos_ids is not None and time_matrix is not None:
            bias = self.combined_bias(pos_ids, pos_ids, time_matrix) #[bs, seqlen, seqlen]
            bias = bias[:, tf.newaxis, :, :] #[bs, 1, seqlen, seqlen]
            atten_scores = atten_scores + bias

        atten_weights = tf.nn.silu(atten_scores)
        atten_weights = self.att_drop(atten_weights, training=training)

        context = tf.matmul(atten_weights, V) ##[bs, numheads, seq, seq] * #[bs, numheads, seq, head_dim] = [bs, numheads, seq, head_dim]
        #合并多头
        context = tf.transpose(context, perm=[0,2,1,3])
        context = tf.reshape(context, (batch_size, seq_len, self.d_model))
        gated_out = U * context #[bs, seq, d_model]

        output = self.out_proj(gated_out)
        output = self.ff_drop(output, training=training)
        output = self.norm2(residual+output)
        
        return output

In [158]:
from tensorflow import keras
from tensorflow.keras import layers
class HSTURecommender(keras.Model):
    def __init__(self, 
                 num_items, 
                 num_actions, 
                 embedding_dim=64, 
                 num_layers=2,
                 num_heads=4,
                 max_seq_len=50,
                 max_time_buckets=64,
                 **kwargs
                ):
        super().__init__(**kwargs)
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len

        self.item_emb = keras.layers.Embedding(num_items + 1, embedding_dim, mask_zero=True)
        self.action_emb = keras.layers.Embedding(num_actions, embedding_dim)
        self.pos_enc = keras.layers.Embedding(max_seq_len * 2, embedding_dim)

        self.hstu_layers = []
        for i in range(num_layers):
            self.hstu_layers.append(
                HSTULayer(
                    d_model = embedding_dim, 
                    num_heads = num_heads, 
                    max_position = max_seq_len,
                    max_time_buckets = max_time_buckets, 
                    dropout_rate = 0.1,
                    name=f"hstu_layer_{i}"
                )
            )
        # """
        # 输出层（权重绑定）共享了输入和输出层的参数，节省内存，加速训练。
        # 使模型具有清晰的线性解释：用户隐状态与物品嵌入的点积即为分数。
        # """
        # self.item_emb.build((None,))
        # self.output_p = layers.Dense(num_items, use_bias=False)
        # self.output_p.build([None, embedding_dim])
        # # 将输出层的权重设置为物品嵌入矩阵（去掉 [MASK] 位）
        # self.output_p.kernel = self.item_emb.embeddings[1:]
        # #error: property 'kernel' of 'Dense' object has no setter

    def compute_time_matrix(self, timestamps):
        return timestamps[:, :, tf.newaxis] - timestamps[:, tf.newaxis, :]

    def call(self, inputs, training=False, return_state=False):
        items = inputs["items"]
        actions = inputs["actions"]
        times = inputs["times"]
        batch_size = tf.shape(items)[0]
        seq_len =  tf.shape(items)[1]

        item_emb = self.item_emb(items) #[bs, seqlen, emb_dim]
        action_emb = self.action_emb(actions)
        seq_emb = item_emb + action_emb

        positions = tf.range(seq_len)[tf.newaxis,:] #[1, seqlen]
        pos_emb = self.pos_enc(positions)
        seq_emb += pos_emb
        
        pos_ids = tf.range(seq_len)[tf.newaxis,:]
        pos_ids = tf.tile(pos_ids, [batch_size, 1]) #[bs, seqlen]
        time_matrix = self.compute_time_matrix(times) 

        x = seq_emb
        for layer in self.hstu_layers:
            x = layer(x, pos_ids=pos_ids,time_matrix=time_matrix,training=training) ##[bs, seq, d_model]
        
        if return_state:
            return x[:, -1, :] #取最后一个位置的隐状态（用于排序优化）[bs, d_model]
        #logits = self.output_p(x) #[bs, seq_len, num_items]
        # 直接使用物品嵌入矩阵计算 logits
        # 物品嵌入形状: (num_items+1, embedding_dim)，去掉 [MASK] 索引0，然后转置为 (embedding_dim, num_items)
        item_embeddings = self.item_emb.embeddings[1:, :]    # (num_items, embedding_dim)
        # x 形状: (batch, seq_len, embedding_dim)
        logits = tf.matmul(x, item_embeddings, transpose_b=True)  # (batch, seq_len, num_items)
        return logits

    def get_final_state(self, inputs):
        ##取最后一个位置的隐状态（用于M-FALCON排序优化）
        return self.call(inputs, training=False, return_state=True)
        
            

In [147]:
model = HSTURecommender(
    num_items=num_items, 
    num_actions=2, 
    embedding_dim=64, 
    num_layers=2,
    num_heads=4,
    max_seq_len=50,
    max_time_buckets=64)

In [84]:
x = np.array([1,0,1,0,1,2])
x=x[:,tf.newaxis]
y=tf.equal(x, [1,2])
print(y)
z = tf.reduce_any(y, axis=1)
#z = tf.cast(y, tf.float32)
# a = tf.reduce_sum(x)
print(z)

# x = np.array([1,0,1,0,1])
# x=x[:,tf.newaxis]
# y=tf.reduce_any(tf.equal(x, [1]),axis=1)
# e = tf.cast(y, tf.float32)
# print(e)

tf.Tensor(
[[ True False]
 [False False]
 [ True False]
 [False False]
 [ True False]
 [False  True]], shape=(6, 2), dtype=bool)
tf.Tensor([ True False  True False  True  True], shape=(6,), dtype=bool)


In [148]:
def masked_sequence_loss(y_true, y_pred, positive_actions=[1]):
    target_items = y_true['target_items']
    target_actions = y_true['target_actions']

    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=None)
    losses = loss_fn(target_items, y_pred)

    #只对正向动作计算损失
    mask = tf.cast(tf.reduce_any(tf.equal(target_actions[:, tf.newaxis], positive_actions),axis=1), tf.float32)
    masked_loss = losses * mask
    avg_loss = tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-9)
    return avg_loss
    
    

In [150]:
optimizer = keras.optimizers.Adam(0.001)

@tf.function 
def train_step(inputs, targets):
    with tf.GradientTape() as tape:
        logits = model(inputs, training = True)
        last_logits = logits[:, -1, :] #取最后一个位置的预测
        loss = masked_sequence_loss(targets, last_logits)
    grad = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grad, model.trainable_variables))
    return loss

@tf.function 
def val_step(inputs, targets):
    logits = model(inputs, training = False)
    last_logits = logits[:, -1, :]
    loss = masked_sequence_loss(targets, last_logits)
    return loss

In [151]:
best_val_loss = float('inf')
best_epoch = 0
epochs = 10
for epoch in range(epochs):
    train_loss = 0.0
    for batch in train_ds:
        inputs, targets = batch
        loss = train_step(inputs, targets)
        train_loss += loss
    avg_train_loss = train_loss / len(train_ds)

    val_loss = 0.0
    for batch in val_ds:
        inputs,targets = batch
        loss = val_step(inputs, targets)
        val_loss += loss
    avg_val_loss = val_loss / len(val_ds)
    print(f"epoch {epoch}/{epochs}-train loss:{avg_train_loss:.4f}-val loss:{avg_val_loss:.4f}")

    #根据val选择最优epoch
    if avg_val_loss < best_val_loss and best_val_loss - avg_val_loss > 0.01:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        model.save_weights('hstu_model.weights.h5')
        print(f"save best model epoch{best_epoch}")

print(f"train finished, best model at epoch {best_epoch}, val loss {best_val_loss}")

epoch 0/10-train loss:8.0205-val loss:7.6818
save best model epoch0
epoch 1/10-train loss:7.4768-val loss:7.4896
save best model epoch1
epoch 2/10-train loss:7.1960-val loss:7.4169
save best model epoch2
epoch 3/10-train loss:6.9342-val loss:7.3956
save best model epoch3
epoch 4/10-train loss:6.6618-val loss:7.4921
epoch 5/10-train loss:6.4061-val loss:7.6448
epoch 6/10-train loss:6.1493-val loss:7.7713
epoch 7/10-train loss:5.8924-val loss:7.9964
epoch 8/10-train loss:5.6363-val loss:8.2329
epoch 9/10-train loss:5.3877-val loss:8.4990
train finished, best model at epoch 3, val loss 7.39561653137207


In [152]:
model.load_weights('hstu_model.weights.h5')

def hit_rate_at_k(model, dataset, k=10):
    total = 0
    hits = 0
    for batch in test_ds:
        inputs,targets = batch
        logits = model(inputs, training=False) #[bs, seq, num_items]
        last_logits = logits[:,-1,:] #[bs, num_items]
        top_k = tf.math.top_k(last_logits,k).indices.numpy() # topk max value's indices(索引)
        target_items = targets['target_items'].numpy() #(bs,)
        for i in range(len(target_items)): #bs
            if target_items[i] in top_k[i]:
                hits += 1
        total += len(target_items)
    return hits / total

In [153]:
test_hit = hit_rate_at_k(model, test_ds, k=10)
print(f'test HR@10:{test_hit:.4f}')

test HR@10:0.0581


In [155]:
#大数据集排序推理时用隐状态 + M‑FALCON 加速
def recommend_mfalon(model, inputs, candidates=None, top_k=10):
    user_state = model.get_final_state(inputs) #[bs, d_model]

    if candidate_ids is None:
        candidate_ids = tf.range(1, model.num_items, dtype=tf.int32)
    else:
        candidate_ids = tf.convert_to_tensor(candidate_ids, dtype=tf.int32)
    cand_embs = model.item_embeddings(candidate_ids) #[num_cand, d_model]
    scores = tf.matmul(user_state, cand_embs, transpose_b = True)
    top_scores, top_indices = tf.math.top_k(scores, top_k)
    top_items = tf.gather(candidate_ids, top_indices) #从候选集选出top k个
    return top_items.numpy(), top_scores.numpy()